## 1. Importación de Librerías y Configuración
En este bloque cargamos las librerías necesarias para la manipulación de datos y la gestión de archivos. 
Utilizamos `pandas` para el procesamiento de tablas y `numpy` para el manejo de valores nulos.

In [69]:
import pandas as pd
import numpy as np
import os

# Configuración de rutas absolutas dinámicas
# Cuando se ejecuta via main.py, el CWD puede variar, así que calculamos la raíz del proyecto
CURRENT_DIR = os.getcwd()
if os.path.basename(CURRENT_DIR) == 'notebooks':
    BASE_DIR = os.path.dirname(CURRENT_DIR)
else:
    BASE_DIR = CURRENT_DIR

print(f"Librerías cargadas. Base del proyecto: {BASE_DIR}")

Librerías cargadas. Base del proyecto: d:\Trabajos_Proyectos\LODO\cargadb


## 2. Carga del Dataset y Detección de Cabeceras
Leemos el archivo descargado por el script de Ingestión. 
Implementamos una lógica para saltar filas vacías o metadatos iniciales que suelen venir en los Excels 
buscando una fila que contenga palabras clave como 'nombre' o 'startup'.

In [70]:
# Configuración de ruta hacia el archivo estandarizado
ruta_archivo = os.path.join(BASE_DIR, "data", "1_sucios", "dataset_para_limpiar.xlsx")

try:
    if os.path.exists(ruta_archivo):
        # Carga inicial para inspección
        df_raw = pd.read_excel(ruta_archivo)
        
        # Palabras clave definidas para localizar la fila de títulos
        header_row = 0
        keywords = ['name', 'website', 'vertical', 'sub vertical', 'descripción']
        
        # Búsqueda de la fila de encabezados en los primeros 15 registros
        for i, row in df_raw.head(15).iterrows():
            # Convertimos toda la fila a una cadena de texto en minúsculas para comparar
            fila_texto = " ".join([str(val).lower() for val in row.values if not pd.isna(val)])
            if any(kw in fila_texto for kw in keywords):
                header_row = i + 1
                break
                
        # Lectura definitiva aplicando el salto de filas
        df = pd.read_excel(ruta_archivo, skiprows=header_row)
        
        # Limpieza de nombres de columnas (quitar espacios extra o saltos de línea)
        df.columns = [str(col).strip() for col in df.columns]
        
        print("Archivo cargado y encabezados identificados.")
        print(f"Columnas detectadas: {list(df.columns)}")
        print(f"Total de registros: {len(df)}")
        
        # Visualización de las primeras 5 filas
        print("\nVista previa de datos:")
        display(df.head(5)) 
    else:
        print(f"Error: Archivo no encontrado en {ruta_archivo}")

except Exception as e:
    print(f"Error en el procesamiento: {e}")

Archivo cargado y encabezados identificados.
Columnas detectadas: ['name', 'website', 'vertical', 'sub vertical', 'Descripcion / solución', 'país ¿dónde se encuentra su principal sede de operaciones?', 'ciudad/región', 'Estadío actual', 'Teléfono', 'Mail', 'LINKEDIN', 'INSTAGRAM', 'FACEBOOK', 'X (twitter)', 'Logo', 'Founder/s', 'Founded', 'Modelo de Negocio', 'DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?"', 'Funding total 2026', 'ETAPA DE MADUREZ ¿En qué momento del crecimiento está?', 'ÁREA INFLUENCIA: ¿qué alcance tiene su solución hoy?', 'ESTADIO SOLUCIÓN (¿En qué nivel de madurez tecnológica se encuentra tu proyecto?)"', 'IMPACTO /SOCIOAMBIENTAL', 'RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?']
Total de registros: 3035

Vista previa de datos:


,name,website,vertical,sub vertical,Descripcion / solución,país ¿dónde se encuentra su principal sede de operaciones?,ciudad/región,Estadío actual,Teléfono,Mail,...,Founder/s,Founded,Modelo de Negocio,"DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?""",Funding total 2026,ETAPA DE MADUREZ ¿En qué momento del crecimiento está?,ÁREA INFLUENCIA: ¿qué alcance tiene su solución hoy?,"ESTADIO SOLUCIÓN (¿En qué nivel de madurez tecnológica se encuentra tu proyecto?)""",IMPACTO /SOCIOAMBIENTAL,RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?
0,Spoiler Alert,https://www.spoileralert.com/?ref=failory,OTRO,Transversal al sector,El software que permite a los socios comercial...,Estados Unidos,Bostón,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HyProMag,https://hypromag.com/?ref=failory,CIRCULAR ECONOMY,recycling,La tecnología principal que comercializa HyPro...,Reino Unido,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DePoly,https://www.depoly.co/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,Empresa de reciclaje químico de plásticos que ...,Suiza,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Glacier,https://endwaste.io/?ref=failory,CIRCULAR ECONOMY,Waste Valorization,construimos robots de última generación que h...,Estados Unidos,San Francisco,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,800 Super Holdings,https://www.800super.com.sg/?ref=failory,CIRCULAR ECONOMY,recycling,Ofrecen soluciones ambientales para organizaci...,Singapur,Sembawang,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Homologación de Nombres de Columnas
En este paso se renombran las columnas del Excel utilizando los títulos exactos detectados en el archivo sucio. 
Se mapean hacia los nombres técnicos de la tabla `organizations` para garantizar la compatibilidad con la base de datos.

In [71]:
# 1. Diccionario de mapeo: 'Columna Excel': 'Columna Técnica/Temporal'
mapping_dict = {
    'name': 'name',
    'website': 'website',
    'vertical': 'vertical',
    'sub vertical': 'sub_vertical',
    'Descripcion / solución': 'solucion',
    'país ¿dónde se encuentra su principal sede de operaciones?': 'country',
    'ciudad/región': 'region', # Mapeamos a region inicialmente
    'Estadío actual': 'estadio_actual',
    'Teléfono': 'contact_phone',
    'Mail': 'mail',
    'Founder/s': 'founders',
    'Founded': 'founded',
    'Modelo de Negocio': 'business_model',
    'DESTACADO/ BADGES ¿QUÉ LA HACE ESPECIAL?': 'badges',
    'RESULTADO / OUTCOME ¿QUÉ PASÓ CON LA EMPRESA?': 'outcome_status',
    'LINKEDIN': 'linkedin_url',
    'INSTAGRAM': 'instagram_url',
    'FACEBOOK': 'facebook_url',
    'X (twitter)': 'twitter_url',
    'Logo': 'logo_url'
}

# Aplicar el renombrado inicial
df_mapeado = df.rename(columns=mapping_dict)

# 2. Creación manual de 'city' (basada en la columna de región para limpieza posterior)
# Esto permite que en el Paso 4 puedas separar o limpiar ciudad y región por separado
df_mapeado['city'] = df_mapeado['region'].copy()

# 3. Definición de la estructura de la base de datos (según Migration)
columnas_db_final = [
    'id', 'name', 'website', 'vertical', 'sub_vertical', 'location', 
    'logo_url', 'estadio_actual', 'solucion', 'mail', 'social_media',
    'contact_phone', 'founders', 'founded', 'organization_type', 'outcome_status', 
    'business_model', 'badges', 'notes', 'status', 'lat', 'lng'
]

# 4. Columnas temporales para el Paso 4 de limpieza
columnas_temporales = [
    'country', 'region', 'city', 
    'linkedin_url', 'instagram_url', 'facebook_url', 'twitter_url'
]

# 5. Garantizar que todas las columnas existan
todas_las_cols = list(set(columnas_db_final + columnas_temporales))
for col in todas_las_cols:
    if col not in df_mapeado.columns:
        df_mapeado[col] = None

# 6. Generar el DataFrame de trabajo final
df_final = df_mapeado[todas_las_cols].copy()

# 7. Valores por defecto
df_final['status'] = 'DRAFT'
df_final['organization_type'] = 'startup'

print(f"Paso 3 completado: Estructura alineada incluyendo 'city'.")
display(df_final[['name', 'country', 'region', 'city']].head(3))

Paso 3 completado: Estructura alineada incluyendo 'city'.


,name,country,region,city
0,Spoiler Alert,Estados Unidos,Bostón,Bostón
1,HyProMag,Reino Unido,NaN,NaN
2,DePoly,Suiza,NaN,NaN


In [72]:
display(df_final.head(20))

,contact_phone,facebook_url,badges,linkedin_url,name,sub_vertical,outcome_status,instagram_url,social_media,business_model,...,city,lng,status,organization_type,notes,website,solucion,id,founders,location
0,NaN,NaN,None,NaN,Spoiler Alert,Transversal al sector,NaN,NaN,None,NaN,...,Bostón,None,DRAFT,startup,None,https://www.spoileralert.com/?ref=failory,El software que permite a los socios comercial...,None,NaN,None
1,NaN,NaN,None,NaN,HyProMag,recycling,NaN,NaN,None,NaN,...,NaN,None,DRAFT,startup,None,https://hypromag.com/?ref=failory,La tecnología principal que comercializa HyPro...,None,NaN,None
2,NaN,NaN,None,NaN,DePoly,Waste Valorization,NaN,NaN,None,NaN,...,NaN,None,DRAFT,startup,None,https://www.depoly.co/?ref=failory,Empresa de reciclaje químico de plásticos que ...,None,NaN,None
3,NaN,NaN,None,NaN,Glacier,Waste Valorization,NaN,NaN,None,NaN,...,San Francisco,None,DRAFT,startup,None,https://endwaste.io/?ref=failory,construimos robots de última generación que h...,None,NaN,None
4,NaN,NaN,None,NaN,800 Super Holdings,recycling,NaN,NaN,None,NaN,...,Sembawang,None,DRAFT,startup,None,https://www.800super.com.sg/?ref=failory,Ofrecen soluciones ambientales para organizaci...,None,NaN,None
5,NaN,NaN,None,NaN,Greyparrot,recycling,NaN,NaN,None,NaN,...,NaN,None,DRAFT,startup,None,https://www.greyparrot.ai/?ref=failory,Plataforma IA de análisis inteligente de resid...,None,NaN,None
6,NaN,NaN,None,NaN,NoPalm-Ingredients,Agri-food Upcycling,NaN,NaN,None,NaN,...,NaN,None,DRAFT,startup,None,https://www.nopalm-ingredients.com/?ref=failory,Foodtech/biotech que produce alternativas sost...,None,NaN,None
7,NaN,NaN,None,NaN,Infiniti Recycling,recycling,NaN,NaN,None,NaN,...,Cambridge,None,DRAFT,startup,None,https://infinitirecycling.com/?ref=failory,Nuestro enfoque de economía circular se centra...,None,NaN,None
8,NaN,NaN,None,NaN,Ace Green Recycling,recycling,NaN,NaN,None,NaN,...,Houston,None,DRAFT,startup,None,https://www.acegreenrecycling.com/?ref=failory,Una innovadora plataforma tecnológica de recic...,None,NaN,None
9,NaN,NaN,None,NaN,Fourth Partner Energy,recycling,NaN,NaN,None,NaN,...,NaN,None,DRAFT,startup,None,http://fourthpartner.co/?ref=failory,Nos dedicamos a transformar las fuentes de ene...,None,NaN,None


## 4. Alineación de Taxonomías con la Base de Datos
En este bloque se transforman los datos descriptivos del Excel a los valores técnicos oficiales de la base de datos. 
Se procesan las categorías de Verticales, Sub-verticales, Etapa de Madurez, Modelo de Negocio, Badges, Tipo de Organización y Estado Operativo.
Esto asegura que el filtrado en el Frontend y la integridad de la base de datos sean perfectos.

In [73]:
import pandas as pd

# 1. Diccionarios de Taxonomía
mapeos = {
    'vertical': {
        'agtech': 'agtech',
        'biotech': 'biotech_bioinputs', 'bioinsumos': 'biotech_bioinputs', 'biotecnología': 'biotech_bioinputs',
        'foodtech': 'foodtech',
        'climatech': 'climatech', 'clima': 'climatech',
        'circular economy': 'circular_economy', 'economía circular': 'circular_economy', 'circular': 'circular_economy',
        'otra': 'otra'
    },
    'sub_vertical': {
        'digital': 'digital_ag', 'software': 'digital_ag',
        'hardware': 'ag_hardware', 'automatización': 'ag_hardware',
        'riego': 'water_tech', 'agua': 'water_tech',
        'fintech': 'ag_fintech',
        'genómica': 'crop_genomics',
        'biofertilizer': 'biofertilizers', 'biofertilizante': 'biofertilizers',
        'biopesticide': 'biopesticides', 'biopesticida': 'biopesticides',
        'sustainable inputs': 'sustainable_inputs', 'insumos sostenibles': 'sustainable_inputs',
        'genetics': 'genetics_breeding', 'genética': 'genetics_breeding',
        'bioengineering': 'bioengineering', 'bioingeniería': 'bioengineering',
        'novel ingredients': 'novel_ingredients', 'ingredientes': 'novel_ingredients',
        'processing': 'food_processing', 'procesamiento': 'food_processing',
        'indoor': 'indoor_ag',
        'safety': 'food_safety', 'trazabilidad': 'food_safety',
        'carbon': 'carbon_solutions', 'carbono': 'carbon_solutions',
        'soil health': 'soil_health', 'suelo': 'soil_health',
        'waste': 'waste_upcycling', 'residuos': 'waste_upcycling', 'valorization': 'waste_upcycling',
        'otra': 'otra_especificar'
    },
    'estadio_actual': {
        'pre-seed': 'pre_seed', 'pre seed': 'pre_seed',
        'seed': 'seed', 'semilla': 'seed',
        'early stage': 'early_stage',
        'series a': 'series_a_early_growth',
        'series b': 'series_b_growth',
        'scale-up': 'scale_up', 'scaleup': 'scale_up',
        'mature': 'mature_late_stage',
        'acquired': 'acquired', 'adquirida': 'acquired',
        'exit': 'exit',
        'unknown': 'unknown'
    }
}

# 2. Funciones de Limpieza y Normalización
def normalizar_taxonomia(valor, diccionario, default_val):
    if pd.isna(valor) or str(valor).strip() == "":
        return default_val
    limpio = str(valor).lower().strip()
    if limpio in diccionario.values():
        return limpio
    for clave, tecnico in diccionario.items():
        if clave in limpio:
            return tecnico
    return default_val

def limpiar_string(val):
    if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '', 's/d', 'null']:
        return "S/D"
    # .title() para que quede "Santa Fe", "Argentina", etc.
    return str(val).strip().title()

# 3. Lógica de SPLIT para Región y Ciudad
def procesar_ubicacion_compleja(row):
    texto_original = str(row['region']) if pd.notna(row['region']) else ""
    
    # Caso: "Santa Fe, Rosario"
    if ',' in texto_original:
        partes = [p.strip() for p in texto_original.split(',')]
        region_limpia = limpiar_string(partes[0])
        city_limpia = limpiar_string(partes[1])
    else:
        # Caso: "Santa Fe" (Sin coma, city queda S/D)
        region_limpia = limpiar_string(texto_original)
        city_limpia = "S/D"
        
    return region_limpia, city_limpia

# --- EJECUCIÓN DEL PASO 4 ---

# A. Normalizar Verticales y Estadio
df_final['vertical'] = df_final['vertical'].apply(lambda x: normalizar_taxonomia(x, mapeos['vertical'], 'otra'))
df_final['sub_vertical'] = df_final['sub_vertical'].apply(lambda x: normalizar_taxonomia(x, mapeos['sub_vertical'], 'otra_especificar'))
df_final['estadio_actual'] = df_final['estadio_actual'].apply(lambda x: normalizar_taxonomia(x, mapeos['estadio_actual'], 'unknown'))

# B. Normalizar País
df_final['country'] = df_final['country'].apply(limpiar_string)

# C. Ejecutar Split de Region/City
df_final[['region', 'city']] = df_final.apply(
    lambda r: pd.Series(procesar_ubicacion_compleja(r)), axis=1
)

# D. Limpieza de URLs de RRSS
columnas_rrss = ['linkedin_url', 'instagram_url', 'facebook_url', 'twitter_url']
for col in columnas_rrss:
    df_final[col] = df_final[col].apply(lambda x: str(x).strip() if pd.notna(x) and str(x).strip() != '' else None)

print("Paso 4 completado con éxito.")
print("Verificación de ubicaciones procesadas:")
display(df_final[['name', 'country', 'region', 'city']].head(10))

Paso 4 completado con éxito.
Verificación de ubicaciones procesadas:


,name,country,region,city
0,Spoiler Alert,Estados Unidos,Bostón,S/D
1,HyProMag,Reino Unido,S/D,S/D
2,DePoly,Suiza,S/D,S/D
3,Glacier,Estados Unidos,San Francisco,S/D
4,800 Super Holdings,Singapur,Sembawang,S/D
5,Greyparrot,Reino Unido,S/D,S/D
6,NoPalm-Ingredients,Países Bajos,S/D,S/D
7,Infiniti Recycling,Inglaterra,Cambridge,S/D
8,Ace Green Recycling,Estados Unidos,Houston,S/D
9,Fourth Partner Energy,India,S/D,S/D


## 5. Normalización Geográfica Universal
En este bloque se procesa la ubicación de forma dinámica para soportar cualquier país del mundo. 
El script extrae el componente del país de cadenas complejas y estandariza nombres conocidos, 
manteniendo la integridad de los datos para que el servicio de geocoding del backend 
pueda procesar cualquier coordenada global.

In [74]:
import pandas as pd

# 1. Diccionario de Traducción (Alineado con tus necesidades)
mapeo_paises_es = {
    'Switzerland': 'Suiza', 'Usa': 'Estados Unidos', 'United States': 'Estados Unidos',
    'United Kingdom': 'Reino Unido', 'Uk': 'Reino Unido', 'England': 'Reino Unido',
    'Netherlands': 'Países Bajos', 'The Netherlands': 'Países Bajos', 'Germany': 'Alemania',
    'Spain': 'España', 'Brazil': 'Brasil', 'Singapore': 'Singapur', 'France': 'Francia',
    'Finland': 'Finlandia', 'Mexico': 'México'
}

# 2. Diccionario de Inferencia (Si el país es S/D pero conocemos la ciudad)
ciudades_famosas = {
    'Bostón': 'Estados Unidos', 'Boston': 'Estados Unidos', 'San Francisco': 'Estados Unidos',
    'Houston': 'Estados Unidos', 'Cambridge': 'Reino Unido', 'London': 'Reino Unido',
    'Colonia': 'Alemania', 'Sembawang': 'Singapur', 'Rosario': 'Argentina'
}

def motor_geografico_refinado(fila):
    # Recuperamos lo que el Paso 4 dejó (sin unificar nada)
    pais = str(fila['country']).strip()
    region = str(fila['region']).strip()
    city = str(fila['city']).strip()

    # --- A. LIMPIEZA DE RUIDO (Esa "D" o "S" fantasma) ---
    # Si el país empieza con "D " o "S ", lo limpiamos
    if pais.startswith('D ') or pais.startswith('S '):
        pais = pais[2:].strip()

    # --- B. TRADUCCIÓN ---
    if pais in mapeo_paises_es:
        pais = mapeo_paises_es[pais]
    
    # --- C. INFERENCIA ---
    # Si el país es S/D o United Kingdom (para asegurar Reino Unido), inferimos por región
    if pais in ['S/D', 'United Kingdom', '']:
        if region in ciudades_famosas:
            pais = ciudades_famosas[region]
        elif city in ciudades_famosas:
            pais = ciudades_famosas[city]

    # --- D. NORMALIZACIÓN FINAL ---
    # Si después de todo el país sigue siendo United Kingdom, lo pasamos a español
    if pais == 'United Kingdom': pais = 'Reino Unido'
    
    # Si región o ciudad quedaron como "S" o "D" por el error anterior, los reseteamos a S/D
    if region in ['S', 'D']: region = 'S/D'
    if city in ['S', 'D']: city = 'S/D'

    return pais, region, city

# EJECUCIÓN: Aplicamos la mejora sin pisar con lógica de strings
df_final[['country', 'region', 'city']] = df_final.apply(
    lambda r: pd.Series(motor_geografico_refinado(r)), axis=1
)

print("Paso 5 completado: Traducción e inferencia aplicadas sobre columnas atómicas.")
display(df_final[['name', 'country', 'region', 'city']].head(15))

Paso 5 completado: Traducción e inferencia aplicadas sobre columnas atómicas.


,name,country,region,city
0,Spoiler Alert,Estados Unidos,Bostón,S/D
1,HyProMag,Reino Unido,S/D,S/D
2,DePoly,Suiza,S/D,S/D
3,Glacier,Estados Unidos,San Francisco,S/D
4,800 Super Holdings,Singapur,Sembawang,S/D
5,Greyparrot,Reino Unido,S/D,S/D
6,NoPalm-Ingredients,Países Bajos,S/D,S/D
7,Infiniti Recycling,Inglaterra,Cambridge,S/D
8,Ace Green Recycling,Estados Unidos,Houston,S/D
9,Fourth Partner Energy,India,S/D,S/D


## 6. Traducción y Enriquecimiento de Ubicación
Este bloque estandariza los nombres de los países al español y resuelve abreviaturas. 
Además, aplica una lógica de inferencia: si el campo 'country' está vacío pero se dispone 
de una región o ciudad conocida, el sistema asigna automáticamente el país 
correspondiente para asegurar la integridad de los datos de geolocalización.

In [75]:
import pandas as pd

# 1. Diccionario maestro de traducciones (Sin lógica de parsing, solo reemplazo)
mapeo_paises_es = {
    'Switzerland': 'Suiza', 'Usa': 'Estados Unidos', 'United States': 'Estados Unidos',
    'Eeuu': 'Estados Unidos', 'United Kingdom': 'Reino Unido', 'Uk': 'Reino Unido',
    'England': 'Reino Unido', 'Netherlands': 'Países Bajos', 'The Netherlands': 'Países Bajos',
    'Germany': 'Alemania', 'Spain': 'España', 'Brazil': 'Brasil', 'Singapore': 'Singapur',
    'France': 'Francia', 'Italy': 'Italia', 'Belgium': 'Bélgica', 'Mexico': 'México',
    'Israel': 'Israel', 'Australia': 'Australia', 'India': 'India', 'Finland': 'Finlandia'
}

# 2. Diccionario de inferencia (Para cuando country es "S/D")
ciudades_famosas = {
    'San Francisco': 'Estados Unidos', 'New York': 'Estados Unidos', 'Miami': 'Estados Unidos',
    'London': 'Reino Unido', 'Cambridge': 'Reino Unido', 'Madrid': 'España', 
    'Barcelona': 'España', 'Mendoza': 'Argentina', 'Buenos Aires': 'Argentina',
    'Rosario': 'Argentina', 'Cordoba': 'Argentina', 'Santiago': 'Chile',
    'Bostón': 'Estados Unidos', 'Sembawang': 'Singapur', 'Houston': 'Estados Unidos'
}

def motor_geografico_integral(fila):
    # Recuperamos los datos exactos del Paso 4/5
    pais = str(fila['country']).strip()
    region = str(fila['region']).strip()
    ciudad = str(fila['city']).strip()

    # --- A. TRADUCCIÓN ---
    # Si el país está en inglés o tiene errores conocidos, lo traducimos
    pais_tit = pais.title()
    if pais_tit in mapeo_paises_es:
        pais = mapeo_paises_es[pais_tit]
    elif pais == "S/D" or pais == "":
        # --- B. INFERENCIA (Solo si el país no existe) ---
        for loc in [region, ciudad]:
            if loc in ciudades_famosas:
                pais = ciudades_famosas[loc]
                break

    # --- C. PROTECCIÓN DE DATOS ---
    # Aseguramos que nada vuelva a convertirse en "S" o "D" sueltos
    if region in ['S', 'D']: region = "S/D"
    if ciudad in ['S', 'D']: ciudad = "S/D"
    
    # Si el país quedó como "United Kingdom" por algún motivo, forzamos español
    if pais == 'United Kingdom': pais = 'Reino Unido'

    return pais, region, ciudad

# EJECUCIÓN: Aplicamos el motor sobre las columnas existentes
# Usamos una técnica de asignación que no altera el orden de las filas
resultados_geo = df_final.apply(motor_geografico_integral, axis=1)
df_final['country'] = [r[0] for r in resultados_geo]
df_final['region'] = [r[1] for r in resultados_geo]
df_final['city'] = [r[2] for r in resultados_geo]

print("Paso 6 completado: Traducción e Inferencia aplicadas sin alterar la estructura previa.")
display(df_final[['name', 'country', 'region', 'city']].head(10))

Paso 6 completado: Traducción e Inferencia aplicadas sin alterar la estructura previa.


,name,country,region,city
0,Spoiler Alert,Estados Unidos,Bostón,S/D
1,HyProMag,Reino Unido,S/D,S/D
2,DePoly,Suiza,S/D,S/D
3,Glacier,Estados Unidos,San Francisco,S/D
4,800 Super Holdings,Singapur,Sembawang,S/D
5,Greyparrot,Reino Unido,S/D,S/D
6,NoPalm-Ingredients,Países Bajos,S/D,S/D
7,Infiniti Recycling,Inglaterra,Cambridge,S/D
8,Ace Green Recycling,Estados Unidos,Houston,S/D
9,Fourth Partner Energy,India,S/D,S/D


## 7. Normalización de Presencia Digital (Deduplicación)
Este bloque estandariza las URLs del sitio web y redes sociales (LinkedIn, Instagram, etc.). 
Al eliminar protocolos y prefijos innecesarios, se asegura que el Migrador pueda 
detectar duplicados de forma efectiva y que los enlaces sean consistentes en el Admin Panel.

In [76]:
import re
import json

# 1. Función de limpieza profesional de URLs
def limpiar_url_profesional(url):
    # Si es nulo o nuestro "S/D", devolvemos string vacío para no ensuciar el JSON
    if pd.isna(url) or str(url).strip().lower() in ['nan', 'none', '', 's/d']:
        return ""
    
    url_limpia = str(url).lower().strip()
    # Eliminamos protocolos para que la URL sea más corta y limpia
    url_limpia = re.sub(r'^https?://', '', url_limpia)
    url_limpia = re.sub(r'^www\.', '', url_limpia)
    url_limpia = url_limpia.rstrip('/')
    # Cortamos cualquier parámetro de tracking (todo lo que venga después de ?)
    url_limpia = url_limpia.split('?')[0]
    return url_limpia

# 2. Mapeo de columnas internas vs Llaves del JSON
columnas_rrss_mapeo = {
    'linkedin_url': 'linkedin',
    'instagram_url': 'instagram',
    'facebook_url': 'facebook',
    'twitter_url': 'twitter'
}

# Identificamos las columnas que realmente existen para no dar error
columnas_reales = [col for col in columnas_rrss_mapeo.keys() if col in df_final.columns]

# --- EJECUCIÓN ---

# A. Limpiamos la web principal (se queda como columna individual)
if 'website' in df_final.columns:
    df_final['website'] = df_final['website'].apply(limpiar_url_profesional)

# B. Limpiamos las columnas de redes una por una
for col in columnas_reales:
    df_final[col] = df_final[col].apply(limpiar_url_profesional)

# C. Construimos el campo JSON 'social_media'
def construir_social_media_json(row):
    redes = {}
    for col_interna, key_json in columnas_rrss_mapeo.items():
        # Solo agregamos al JSON si la URL existe y no está vacía
        if col_interna in row and row[col_interna] != "":
            redes[key_json] = row[col_interna]
    
    # ensure_ascii=False para que guarde tildes y caracteres especiales correctamente
    return json.dumps(redes, ensure_ascii=False)

df_final['social_media'] = df_final.apply(construir_social_media_json, axis=1)

print(f"Limpieza de URLs completada. Campo JSON 'social_media' generado.")
display(df_final[['name', 'website', 'social_media']].head(10))

Limpieza de URLs completada. Campo JSON 'social_media' generado.


,name,website,social_media
0,Spoiler Alert,spoileralert.com/,{}
1,HyProMag,hypromag.com/,{}
2,DePoly,depoly.co/,{}
3,Glacier,endwaste.io/,{}
4,800 Super Holdings,800super.com.sg/,{}
5,Greyparrot,greyparrot.ai/,{}
6,NoPalm-Ingredients,nopalm-ingredients.com/,{}
7,Infiniti Recycling,infinitirecycling.com/,{}
8,Ace Green Recycling,acegreenrecycling.com/,{}
9,Fourth Partner Energy,fourthpartner.co/,{}


## 8. Limpieza de Fundadores y Año de Fundación
Este bloque procesa la identidad de los fundadores eliminando enlaces y ruido de texto 
para dejar solo nombres limpios. Asimismo, estandariza el año de fundación a un formato 
numérico consistente, corrigiendo las discrepancias entre tipos de datos (String vs Int).

In [77]:
import re
import json

# 1. Función de limpieza de URLs (Mantenemos tu lógica profesional)
def limpiar_url_profesional(url):
    if pd.isna(url) or str(url).strip().lower() in ['nan', 'none', '', 's/d']:
        return ""
    url_limpia = str(url).lower().strip()
    url_limpia = re.sub(r'^https?://', '', url_limpia)
    url_limpia = re.sub(r'^www\.', '', url_limpia)
    url_limpia = url_limpia.rstrip('/').split('?')[0]
    return url_limpia

# 2. Lógica de Founders (Retorna LISTA JSON para la DB)
def transformar_founders_to_json(texto):
    if pd.isna(texto) or str(texto).strip().lower() in ['nan', 'none', '', 's/d']:
        return json.dumps([]) 
    
    # Limpieza de basura y links en nombres
    texto_limpio = re.sub(r'\(http[s]?://.*?\)', '', str(texto))
    texto_limpio = re.sub(r'http[s]?://\S+', '', texto_limpio)
    texto_limpio = texto_limpio.replace('|', ',')
    texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()
    
    nombres = [n.strip().title() for n in texto_limpio.split(',') if n.strip()]
    return json.dumps(nombres, ensure_ascii=False)

# 3. Función para Badges (Retorna LISTA JSON)
def transformar_badges_to_json(texto):
    if pd.isna(texto) or str(texto).strip().lower() in ['nan', 'none', '', 's/d']:
        return json.dumps([])
    
    badges = [b.strip().upper() for b in str(texto).replace('|', ',').split(',') if b.strip()]
    return json.dumps(badges, ensure_ascii=False)

# 4. Lógica de Founded (Fix para evitar el .0)
def transformar_founded(valor):
    if pd.isna(valor) or str(valor).strip().lower() in ['nan', 'none', '', 's/d']:
        return None 
    solo_numeros = re.sub(r'\D', '', str(valor))
    if solo_numeros and len(solo_numeros) >= 4:
        return int(solo_numeros[:4])
    return None

# --- EJECUCIÓN INTEGRADA (Sin tocar country, region o city) ---

# A. Limpieza de Website y Redes (Solo si existen)
columnas_rrss_mapeo = {
    'linkedin_url': 'linkedin', 
    'instagram_url': 'instagram', 
    'facebook_url': 'facebook', 
    'twitter_url': 'twitter'
}

if 'website' in df_final.columns:
    df_final['website'] = df_final['website'].apply(limpiar_url_profesional)

for col_in in columnas_rrss_mapeo.keys():
    if col_in in df_final.columns:
        df_final[col_in] = df_final[col_in].apply(limpiar_url_profesional)

# B. Empaquetado de social_media JSON (Usando las URLs ya limpias)
df_final['social_media'] = df_final.apply(lambda r: json.dumps({
    key: r[col] for col, key in columnas_rrss_mapeo.items() 
    if col in r and r[col] != ""
}, ensure_ascii=False), axis=1)

# C. Transformación de Founders y Badges a JSON
if 'founders' in df_final.columns:
    df_final['founders'] = df_final['founders'].apply(transformar_founders_to_json)

if 'badges' in df_final.columns:
    df_final['badges'] = df_final['badges'].apply(transformar_badges_to_json)

# D. Normalización de Año con fix de Int64 (Adiós al .0)
if 'founded' in df_final.columns:
    df_final['founded'] = df_final['founded'].apply(transformar_founded).astype('Int64')

print("Paso 8 completado: JSONs generados y años corregidos. La geografía se mantuvo intacta.")
display(df_final[['name', 'country', 'region', 'city', 'social_media', 'founders', 'founded']].head(10))

Paso 8 completado: JSONs generados y años corregidos. La geografía se mantuvo intacta.


,name,country,region,city,social_media,founders,founded
0,Spoiler Alert,Estados Unidos,Bostón,S/D,{},[],<NA>
1,HyProMag,Reino Unido,S/D,S/D,{},[],<NA>
2,DePoly,Suiza,S/D,S/D,{},[],<NA>
3,Glacier,Estados Unidos,San Francisco,S/D,{},[],<NA>
4,800 Super Holdings,Singapur,Sembawang,S/D,{},[],<NA>
5,Greyparrot,Reino Unido,S/D,S/D,{},[],<NA>
6,NoPalm-Ingredients,Países Bajos,S/D,S/D,{},[],<NA>
7,Infiniti Recycling,Inglaterra,Cambridge,S/D,{},[],<NA>
8,Ace Green Recycling,Estados Unidos,Houston,S/D,{},[],<NA>
9,Fourth Partner Energy,India,S/D,S/D,{},[],<NA>


## 9. Normalización de Contactos
Este bloque estandariza los correos electrónicos y teléfonos. Se eliminan espacios 
invisibles en los emails y se limpian los números telefónicos de caracteres especiales 
(paréntesis, guiones, espacios), manteniendo el formato numérico puro para 
compatibilidad con sistemas de mensajería.

In [78]:
import re

def resolver_unico_mail(email):
    # Respetamos nulos y nuestro estándar "S/D"
    if pd.isna(email) or str(email).strip().lower() in ['nan', 'none', '', 's/d']:
        return "S/D"
    
    # Separamos por comas, pipes, puntos y coma o espacios
    partes = re.split(r'[|,;\s]+', str(email).strip())
    # Tomamos el primero y lo pasamos a minúsculas
    primer_mail = partes[0].lower()
    
    return primer_mail

def resolver_unico_phone(tel):
    # Respetamos nulos y nuestro estándar "S/D"
    if pd.isna(tel) or str(tel).strip().lower() in ['nan', 'none', '', 's/d']:
        return "S/D"
    
    # Separamos por los caracteres de división comunes
    partes = re.split(r'[|,;]+', str(tel).strip())
    primer_tel = partes[0].strip()
    
    # Mantenemos el '+' si existe y limpiamos el resto de caracteres (paréntesis, guiones)
    prefijo = "+" if primer_tel.startswith('+') else ""
    solo_numeros = re.sub(r'\D', '', primer_tel)
    
    return f"{prefijo}{solo_numeros}" if solo_numeros else "S/D"

# --- EJECUCIÓN (Sin tocar nada de los pasos 4 al 8) ---

if 'mail' in df_final.columns:
    df_final['mail'] = df_final['mail'].apply(resolver_unico_mail)

if 'contact_phone' in df_final.columns:
    df_final['contact_phone'] = df_final['contact_phone'].apply(resolver_unico_phone)

print("Paso 9: Resolución de contactos finalizada. Solo se conserva el contacto principal.")

# Verificación de los cambios
display(df_final[['name', 'mail', 'contact_phone']].head(10))

Paso 9: Resolución de contactos finalizada. Solo se conserva el contacto principal.


,name,mail,contact_phone
0,Spoiler Alert,S/D,S/D
1,HyProMag,S/D,S/D
2,DePoly,S/D,S/D
3,Glacier,S/D,S/D
4,800 Super Holdings,S/D,S/D
5,Greyparrot,S/D,S/D
6,NoPalm-Ingredients,S/D,S/D
7,Infiniti Recycling,S/D,S/D
8,Ace Green Recycling,S/D,S/D
9,Fourth Partner Energy,S/D,S/D


## 10. Deduplicación y Estandarización de Taxonomía
En este bloque final, se eliminan registros duplicados para asegurar la integridad de 
la base de datos. Además, se aplica la taxonomía 'S/D' a los campos nulos y se 
asigna el tipo 'STARTUP' a todas las organizaciones, dejando el dataset listo 
para la migración final a HeidiSQL.

In [79]:
import json
import uuid

# 1. LIMPIEZA DE MEMORIA: Forzamos que los nulos sean reales antes de armar el JSON
# Esto evita que valores anteriores (como Israel) se propaguen por error
cols_geo = ['country', 'region', 'city']
for col in cols_geo:
    # Si la columna existe, convertimos ruidos y nulos en None real
    if col in df_final.columns:
        df_final[col] = df_final[col].replace(['nan', 'NaN', 'None', '', 'S/D', 'S', 'D', 'D S'], None)

# 2. Función ultra-estricta para el JSON
def construir_location_final(row):
    loc_obj = {}
    
    # Solo si el valor existe y tiene contenido real (más de 1 letra y no es basura)
    for key in ['country', 'region', 'city']:
        val = row[key]
        if pd.notna(val):
            val_str = str(val).strip()
            if len(val_str) > 1 and val_str.upper() not in ['S/D', 'S', 'D', 'D S', 'UNKNOWN']:
                loc_obj[key] = val_str
                
    return json.dumps(loc_obj, ensure_ascii=False)

# 3. EJECUCIÓN: Generamos location y status
df_final['location'] = df_final.apply(construir_location_final, axis=1)

def definir_status_seguro(row):
    # Si el JSON es "{}" (vacío), es DRAFT sí o sí
    return 'IN_REVIEW' if row['location'] != '{}' else 'DRAFT'

df_final['status'] = df_final.apply(definir_status_seguro, axis=1)

# 4. IDs y Estructura de Exportación
df_final['id'] = [str(uuid.uuid4()) for _ in range(len(df_final))]
df_final['organization_type'] = 'startup'

columnas_db = [
    'id', 'name', 'website', 'vertical', 'sub_vertical', 'location', 
    'logo_url', 'estadio_actual', 'solucion', 'mail', 'social_media',
    'contact_phone', 'founders', 'founded', 'organization_type', 'outcome_status', 
    'business_model', 'badges', 'notes', 'status', 'lat', 'lng'
]

# Asegurar columnas y crear df_export
for col in columnas_db:
    if col not in df_final.columns:
        df_final[col] = None

df_export = df_final[columnas_db].copy()

print(f"Limpieza de propagación terminada.")
print(f"Registros en REVISION: {len(df_export[df_export['status'] == 'IN_REVIEW'])}")
print(f"Registros en DRAFT: {len(df_export[df_export['status'] == 'DRAFT'])}")

display(df_export[['name', 'location', 'status']].iloc[2635:2645]) # Verificamos la zona del error

Limpieza de propagación terminada.
Registros en REVISION: 1731
Registros en DRAFT: 1304


,name,location,status
2635,YEVVO,"{""country"": ""Israel""}",IN_REVIEW
2636,Nation - E,"{""country"": ""Israel""}",IN_REVIEW
2637,Bengigi Studio,"{""country"": ""Israel""}",IN_REVIEW
2638,Sovevos,"{""country"": ""Israel""}",IN_REVIEW
2639,StickyTacs,"{""country"": ""Israel""}",IN_REVIEW
2640,Bisconi Cones,"{""country"": ""Israel""}",IN_REVIEW
2641,Zadara Storage,"{""country"": ""Israel""}",IN_REVIEW
2642,iceSCRM,"{""country"": ""Israel""}",IN_REVIEW
2643,Motinize,"{""country"": ""Israel""}",IN_REVIEW
2644,SOOMLA,"{""country"": ""Israel""}",IN_REVIEW


In [ ]:
# 1. Lista maestra de columnas de la BD
columnas_db_reales = [
    'id', 'name', 'website', 'vertical', 'sub_vertical', 'location', 
    'logo_url', 'estadio_actual', 'solucion', 'mail', 'social_media',
    'contact_phone', 'founders', 'founded', 'organization_type', 'outcome_status', 
    'business_model', 'badges', 'notes', 'status', 'lat', 'lng'
]

# 2. Definir qué columnas son técnicas (pueden ser NULL) y cuáles son texto (deben ser S/D)
columnas_tecnicas = ['lat', 'lng', 'founded']

# 3. Filtrar y limpiar nulos prohibidos
df_final = df_final[[col for col in columnas_db_reales if col in df_final.columns]].copy()

for col in df_final.columns:
    if col not in columnas_tecnicas:
        # Si la columna es de texto y tiene un nulo, le ponemos 'S/D'
        # Esto evita el error "Column X cannot be null"
        df_final[col] = df_final[col].fillna('S/D')
        # También convertimos posibles strings 'None' o vacíos en 'S/D'
        df_final[col] = df_final[col].replace(['None', 'nan', 'NaN', ''], 'S/D')

print("Alineación completada. Se han reemplazado los nulos de texto por 'S/D' para evitar errores de integridad.")

Dataset alineado: Se conservan 22 columnas de las 22 requeridas.
Columnas descartadas (no van a la BD): []


,id,name,website,vertical,sub_vertical,location,logo_url,estadio_actual,solucion,mail,...,founders,founded,organization_type,outcome_status,business_model,badges,notes,status,lat,lng
0,a2267cfd-78ca-4845-becd-9f8ef6a7cb0d,Spoiler Alert,spoileralert.com,otra,otra_especificar,"{""country"": ""Estados Unidos"", ""region"": ""Bostón""}",NaN,unknown,El software que permite a los socios comercial...,S/D,...,[],<NA>,startup,NaN,NaN,[],None,IN_REVIEW,None,None
1,f4cf3377-fe6d-4a5e-b5c4-a1b41f7f24c8,HyProMag,hypromag.com,circular_economy,otra_especificar,"{""country"": ""Reino Unido""}",NaN,unknown,La tecnología principal que comercializa HyPro...,S/D,...,[],<NA>,startup,NaN,NaN,[],None,IN_REVIEW,None,None
2,11d9c791-7ce2-4707-8d82-619f01c4d3c8,DePoly,depoly.co,circular_economy,waste_upcycling,"{""country"": ""Suiza""}",NaN,unknown,Empresa de reciclaje químico de plásticos que ...,S/D,...,[],<NA>,startup,NaN,NaN,[],None,IN_REVIEW,None,None
3,b4954694-c28b-46a4-8d54-a6bdba8f4954,Glacier,endwaste.io,circular_economy,waste_upcycling,"{""country"": ""Estados Unidos"", ""region"": ""San F...",NaN,unknown,construimos robots de última generación que h...,S/D,...,[],<NA>,startup,NaN,NaN,[],None,IN_REVIEW,None,None
4,0e83eeaa-9390-41c1-83d0-973ebb463f5b,800 Super Holdings,800super.com.sg,circular_economy,otra_especificar,"{""country"": ""Singapur"", ""region"": ""Sembawang""}",NaN,unknown,Ofrecen soluciones ambientales para organizaci...,S/D,...,[],<NA>,startup,NaN,NaN,[],None,IN_REVIEW,None,None


## 11. Exportación de Datos Limpios
Este bloque genera el archivo final en la ruta 'data/02_limpios/'. Se utiliza 
codificación UTF-8 para preservar caracteres especiales y se asegura que el 
índice de Pandas no se incluya, dejando el archivo listo para la inyección 
directa en la tabla 'organizations'.

In [82]:
import os

# 1. Definir la ruta de salida absoluta
ruta_salida = os.path.join(BASE_DIR, "data", "2_limpios", "startups_limpias_final.csv")

# 2. Asegurar que la carpeta exista
os.makedirs(os.path.dirname(ruta_salida), exist_ok=True)

# 3. Exportar a CSV
df_final.to_csv(ruta_salida, index=False, sep=';', encoding='utf-8-sig')

print(f"¡Proceso completado con éxito!")
print(f"Archivo guardado en: {ruta_salida}")
print(f"Total de registros exportados: {len(df_final)}")

¡Proceso completado con éxito!
Archivo guardado en: d:\Trabajos_Proyectos\LODO\cargadb\data\2_limpios\startups_limpias_final.csv
Total de registros exportados: 3035
